# Exploratory Data Analysis for `php9xWOpn.arff`

This notebook performs a basic EDA workflow on the ARFF dataset in this folder.

It covers:
- data loading and schema inspection
- missing values and duplicates
- class balance
- descriptive statistics
- feature distributions
- correlation analysis
- simple class-wise comparisons


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.io import arff

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11


In [ ]:
dataset_path = Path("php9xWOpn.arff")
data, meta = arff.loadarff(dataset_path)
df = pd.DataFrame(data)

# Decode byte-string nominal columns returned by scipy's ARFF loader.
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].str.decode("utf-8")

# Keep the target readable for plots and summaries.
if "Class" in df.columns:
    df["Class"] = df["Class"].astype("category")

df.head()

In [ ]:
print(f"Dataset path: {dataset_path.resolve()}")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nColumn dtypes:")
print(df.dtypes)

print("\nARFF relation:", meta.name)
print("ARFF attributes:")
for name, spec in meta.items():
    print(f"- {name}: {spec}")

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=False)
})

summary

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows}")

df.describe(include="all").transpose()

## Target Variable

In [ ]:
class_counts = df["Class"].value_counts().sort_index()
class_pct = (df["Class"].value_counts(normalize=True).sort_index() * 100).round(2)

display(pd.DataFrame({"count": class_counts, "percent": class_pct}))

ax = sns.countplot(data=df, x="Class", hue="Class", palette="Set2", legend=False)
ax.set_title("Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.show()

## Numeric Features

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_df = df[numeric_cols]

print(f"Numeric feature count: {len(numeric_cols)}")
numeric_df.describe().transpose()

In [ ]:
# Histograms for all numeric features.
n_cols = 4
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], kde=True, bins=30, ax=ax, color="#4C78A8")
    ax.set_title(col)

for ax in axes[len(numeric_cols):]:
    ax.axis("off")

fig.suptitle("Feature Distributions", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Boxplots can highlight heavy skew and outliers.
n_cols = 4
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for ax, col in zip(axes, numeric_cols):
    sns.boxplot(x=df[col], ax=ax, color="#72B7B2")
    ax.set_title(col)

for ax in axes[len(numeric_cols):]:
    ax.axis("off")

fig.suptitle("Boxplots for Numeric Features", y=1.02)
fig.tight_layout()
plt.show()

## Correlations

In [ ]:
corr = numeric_df.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

corr_pairs.head(15).to_frame("correlation")

## Class-Wise Comparisons

In [ ]:
# Rank numeric features by absolute mean difference between classes.
class_means = df.groupby("Class", observed=False)[numeric_cols].mean()
mean_diff = (class_means.loc[class_means.index[0]] - class_means.loc[class_means.index[1]]).abs().sort_values(ascending=False)
top_features = mean_diff.head(8).index.tolist()

mean_diff.head(15).to_frame("absolute_mean_difference")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.reshape(-1)

for ax, col in zip(axes, top_features):
    sns.boxplot(data=df, x="Class", y=col, hue="Class", palette="Set2", legend=False, ax=ax)
    ax.set_title(f"{col} by Class")

for ax in axes[len(top_features):]:
    ax.axis("off")

fig.suptitle("Top Features by Class Separation", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
class_profile = df.groupby("Class", observed=False)[numeric_cols].agg(["mean", "median", "std"])
class_profile

## Quick Takeaways

After running the notebook, use this section to summarize:
- whether the classes are imbalanced
- which variables appear heavily skewed or outlier-prone
- which features are most strongly correlated
- which variables separate Class `1` and Class `2` most clearly
- whether scaling or transformation may help later modeling
